# 02 - Preparing and running Cloudy calculations

This tutorial shows how to:

1. Prepare a Starburst99 continuum for Cloudy.
2. Construct heating/cooling and ion fractions parameter files.
3. Inspect the resulting configuration
4. Execute Cloudy with pyCIAO.


The radiation spectrum is obtained from an existing Starburst99 model. The physical state of the gas is subsequently calculated by Cloudy.

We deliberately use a small number density-temperature grid to demonstrate the workflow.

In [1]:
# Load packages
import sys
import subprocess
from pathlib import Path
from contextlib import chdir

from py4radiation import SED, ParameterFiles

# Directory for this tutorial's files and Cloudy results
RUN_NAME = "tutorial"
WORKDIR  = (Path.cwd() / "tutorial_runs" / RUN_NAME).resolve()
WORKDIR.mkdir(parents=True, exist_ok=True)

# Modify these values for your system
CLOUDY_EXE = "/home/cphys/software/c13.05/source/cloudy.exe"
SB99_PATH = None

# Set to True when Cloudy is installed and configured
EXECUTE_CLOUDY = True

# Numbre of processors
# We recommend this matches the number of points in the number density grid
NPROC = 2

## 1. Prepare the incident spectrum

We use Starburst99 Figure 8b, with a continuous star-formation rate of 1 $M_{\odot}\,{\rm yr}^{-1}$ with a Salpeter initial mass function ($\alpha=2.35$). The stellar population age is fixed at 3 Myr and the distance is 1 kpc. The spectrum is scaled and converted into the Cloudy-readable format.

In [2]:
with chdir(WORKDIR):
    sed = SED(
        run_name=RUN_NAME,
        table_id="8b",
        sfr_msun_yr=1.0,
        distance_kpc=1.0,
        age_myr=3,
        sb99_path=SB99_PATH,
    )

    sed_file = sed.write_outfile()

print("Prepared continuum:", sed_file)

Prepared continuum: tutorial_z0.0000e+00.out


## 2. Define the Cloudy parameter grids

We determine:

- Hydrogen number densities from $10^{-3}$ to $10^{-2}$ ${\rm cm}^{-3}$, using 2 points.
- Temperatures from $10^{3}$ to $10^{5}$ ${\rm K}$, using 11 points.
- Hydrogen, carbon and oxygen for the ion tables.

The heating and cooling coefficients will be normalised by $n_{\rm H}^2$.

These settings are illustrative, production calculations will generally require more extensive parameter grids.

In [3]:
with chdir(WORKDIR):
    par = ParameterFiles(
        cloudy_path=CLOUDY_EXE, # path to the Cloudy executable
        run_name=RUN_NAME,      # name of the model
        elements="H C O",       # chemical elements, supported up to Zn
        normalisation=1,        # (1) n_H^2; (2) n_e * n_H
        n_limits=(-3, -2),      # hydrogen number density grid boundaries in log10 space
        n_spacing=1.0,          # step size in the logarithmic number density grid
        T_limits=(3, 5),        # temperature grid boundaries in log10 space
        T_points=11,            # number of points in the temperature grid
    )

    # Generate the parameter files for heating/cooling (hc)
    # and ion fractions (ib)
    hc_file = par.hc_parfile()
    ib_file = par.ib_parfile()

print("Heating/cooling parameters:", hc_file)
print("Ionisation parameters:", ib_file)

Heating/cooling parameters: /home/cphys/repos/py4radiation/notebooks/tutorial_runs/tutorial/tutorial_hc.par
Ionisation parameters: /home/cphys/repos/py4radiation/notebooks/tutorial_runs/tutorial/tutorial_ib.par


In [4]:
# Inspect the parameter files

print("=== Heating/cooling parameter file ===\n")
print(hc_file.read_text())

print("\n=== Ionisation parameter file ===\n")
print(ib_file.read_text())

=== Heating/cooling parameter file ===

#####################################################
################## RUN  PARAMETERS ##################

# path to CLOUDY executable
cloudyExe              = /home/cphys/software/c13.05/source/cloudy.exe

# save raw output from CLOUDY
saveCloudyOutputFiles   = 0

# exit if CLOUDY crashes
exitOnCrash             = 1

# run name
outputFilePrefix       = tutorial

# output path
outputDir              = /home/cphys/repos/py4radiation/notebooks/tutorial_runs/tutorial/hc

# index of first run
runStartIndex           = 1

# TEST
test                    = 0

# run mode
cloudyRunMode          = 1

#####################################################
########## HEATING & COOLING MAP PARAMETERS #########

# min T
coolingMapTmin = 1e3

# max T
coolingMapTmax = 1e5

# T resolution (log points)
coolingMapTpoints = 11

# scale factor (1 - n_H^2, 2 - n_H * n_e)
coolingScaleFactor = 1

#####################################################
###################

## 3. Execute the calculations with pyCIAO

pyCIAO distributes the CLoudy calculations across the selected worker processes.

Two configurations with the same incident spectrum and physical grid are executed independently:

- Heating and cooling calculations (mode 1).
- Ion fraction calculations (mode 3)

(The mode numbers correspond to backward compatibility with CIAOLoop (B. Smith et al. 2008)).

The calculations require a working Cloudy executable.

In [5]:
if EXECUTE_CLOUDY:
    cloudy_path = Path(CLOUDY_EXE)

    if not cloudy_path.is_file():
        raise FileNotFoundError(f"Cloudy executable not found: {cloudy_path}")

    for parfile in (hc_file, ib_file):
        command = [
            sys.executable,
            "-m",
            "py4radiation.pyciao.pyciao",
            str(parfile),
            "--nproc",
            str(NPROC),
        ]

        print("Executing:", " ".join(command))

        subprocess.run(command, cwd=WORKDIR, check=True)
else:
    print("Cloudy execution skipped. Set EXECUTE_CLOUDY=True when ready.")

Executing: /home/cphys/miniforge3/envs/py4radiation/bin/python -m py4radiation.pyciao.pyciao /home/cphys/repos/py4radiation/notebooks/tutorial_runs/tutorial/tutorial_hc.par --nproc 2
2026-11-22 02:00:26,050 - INFO - Log file initialised at /home/cphys/repos/py4radiation/notebooks/tutorial_runs/tutorial/hc/tutorial.log.
2026-11-22 02:00:26,050 - INFO - --- pyCIAO Multiprocessing Run Started ---
2026-11-22 02:00:26,050 - INFO - Generated 2 Cloudy tasks.
2026-11-22 02:00:26,050 - INFO - Writing run file: /home/cphys/repos/py4radiation/notebooks/tutorial_runs/tutorial/hc/tutorial.run.
2026-11-22 02:00:26,051 - INFO - Starting multiprocessing pool with 2 processes.
[Worker-265444] INFO - Run 1 STARTED.
[Worker-265445] INFO - Run 2 STARTED.
[Worker-265445] INFO - Run 2 FINISHED in 23.41 s.
[Worker-265444] INFO - Run 1 FINISHED in 23.46 s.
2026-11-22 02:00:49,531 - INFO - --- Run Finished ---
2026-11-22 02:00:49,531 - INFO - Total tasks: 2
2026-11-22 02:00:49,531 - INFO - Successful:  2
2026-

In [6]:
# Inspect the outputs

for name in ("hc", "ib"):
    output_dir = WORKDIR / name

    print(f"\n{name.upper()} outputs:")

    if output_dir.exists():
        for path in sorted(output_dir.iterdir()):
            print(" ", path.name)
    else:
        print("  No calculations have been executed.")


HC outputs:
  tutorial.log
  tutorial.run
  tutorial_run1.dat
  tutorial_run2.dat

IB outputs:
  .ipynb_checkpoints
  tutorial.log
  tutorial.run
  tutorial_iontable.h5
  tutorial_run1_C.dat
  tutorial_run1_H.dat
  tutorial_run1_O.dat
  tutorial_run2_C.dat
  tutorial_run2_H.dat
  tutorial_run2_O.dat


## 4. Understanding the output files

pyCIAO creates a master `.run` file and maps corresponding to the individual Cloudy calculations (one for each value of the number density grid. In the ion-fraction mode, for each number density value there are files for each specified element).

The `.run` file records the ordered parameter combinations and provides the information needed to build multidimensional tables from the results.

The heating/cooling maps contain temperatures, heating and cooling coefficients, and mean molecular weights. The ion fraction maps contain logarithmic temperatures and ion fractions for the requested elements.

The next tutorial processes these outputs.